# Oxideon — Interactive Deep Learning & Scientific Computing Showcase

Run **Oxideon** (`pip install oxideon`) directly in Google Colab or Jupyter — backed by a pure-Rust tensor core with PyTorch-friendly Python APIs:
1. **Zero-copy strided views**, `rearrange` (einops), and NHWC `channels_last` layout
2. **Higher-order autograd** (`create_graph=True`: 1st, 2nd & 3rd derivatives) + differentiable `einsum`
3. **Train a 2D Spiral Classifier** (`Linear` + `LayerNorm` + `AdamW`) with decision boundary & loss plots
4. **CNN & Transformer building blocks** (`Conv2d`, `MultiheadAttention`)
5. **Dynamic quantization** (`quantize_dynamic`) & **SafeTensors** export (`save_file` / `load_file`)
6. **Scientific computing** (`linalg.qr`, `linalg.svd`, `fft.rfft`, `distributions.Normal`)
7. **Zero-copy NumPy / PyTorch DLPack interop** & CPU micro-benchmarks

- **GitHub:** https://github.com/voidstackloop/oxideon
- **Live WASM Browser Demo (no install):** https://voidstackloop.github.io/oxideon/

In [ ]:
!pip install -q oxideon
import oxideon as ox
import math, time, os, tempfile
print(f"Oxideon v{ox.__version__} ready!")

## 1. Zero-Copy Strided Views, Einops `rearrange`, and `channels_last`

In [ ]:
x = ox.Tensor.arange(0.0, 24.0, 1.0).reshape([2, 3, 4])
v = x.transpose(0, 2)
assert v.shares_storage(x)
print("Transposed view shape:", list(v.shape), "| shares_storage:", v.shares_storage(x))

img = ox.randn([2, 8, 4, 4])
nhwc = img.rearrange("b c h w -> b h w c")
cl = img.to_channels_last()
print("Rearranged NHWC shape:", list(nhwc.shape), "| is_channels_last:", cl.is_channels_last())

## 2. Higher-Order Autograd (`create_graph=True`) & `einsum`

In [ ]:
# Analytical 1st, 2nd, and 3rd derivatives of y = x^4 at x = 3
# Expected: dy = 4*27 = 108, d2y = 12*9 = 108, d3y = 24*3 = 72
x = ox.Tensor.from_slice([3.0], [1], requires_grad=True)
y = x.powf(4.0)
dy = ox.grad([y], [x], create_graph=True)[0]
d2y = ox.grad([dy], [x], create_graph=True)[0]
d3y = ox.grad([d2y], [x])[0]
print(f"y=x^4 at x=3 -> dy={dy.to_list()}, d2y={d2y.to_list()}, d3y={d3y.to_list()}")

# Batched Einstein summation
A = ox.Tensor.from_slice([1.0, 2.0, 3.0, 4.0], [2, 2])
B = ox.Tensor.from_slice([2.0, 0.0, 1.0, 2.0], [2, 2])
C = ox.einsum("ij,jk->ik", A, B)
print("einsum matmul [2,2]:", C.to_list())

## 3. Train a Non-Linear 2D Spiral Classifier with `AdamW` & Plot Results

In [ ]:
import numpy as np

def make_spiral(n_per_class=100, seed=42):
    rng = np.random.default_rng(seed)
    xs, ys = [], []
    for cls in (0, 1):
        t = np.linspace(0.05, 1.0, n_per_class)
        r = 0.85 * t + 0.05
        theta = 3.2 * t * math.pi + (math.pi if cls == 1 else 0.0) + rng.normal(0, 0.08, n_per_class)
        pts = np.stack([r * np.sin(theta), r * np.cos(theta)], axis=1)
        xs.append(pts.astype(np.float32))
        ys.append(np.full((n_per_class, 2), [1.0, 0.0] if cls == 0 else [0.0, 1.0], dtype=np.float32))
    return np.concatenate(xs, axis=0), np.concatenate(ys, axis=0)

X_np, Y_np = make_spiral()
X = ox.from_numpy(X_np)
Y = ox.from_numpy(Y_np)
y_cls = Y_np.argmax(axis=1)

class SpiralNet(ox.nn.Module):
    def __init__(self, hid=32):
        super().__init__()
        self.fc1 = ox.nn.Linear(2, hid)
        self.ln1 = ox.nn.LayerNorm(hid)
        self.fc2 = ox.nn.Linear(hid, hid)
        self.ln2 = ox.nn.LayerNorm(hid)
        self.out = ox.nn.Linear(hid, 2)

    def forward(self, x):
        h = self.ln1(self.fc1(x)).tanh()
        h = self.ln2(self.fc2(h)).tanh()
        return self.out(h)

model = SpiralNet(32)
opt = ox.optim.AdamW(model.parameters(), lr=0.03)
losses, accs = [], []

for step in range(180):
    opt.zero_grad()
    probs = model(X).softmax(-1)
    loss = ((probs - Y) * (probs - Y)).mean()
    loss.backward()
    opt.step()
    acc = float((probs.numpy().argmax(axis=1) == y_cls).mean())
    losses.append(loss.item())
    accs.append(acc)
    if step in (0, 59, 119, 179):
        print(f"step {step:3d}: mse_prob={losses[-1]:.5f} | accuracy={acc * 100:.1f}%")

try:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
    gx, gy = np.meshgrid(np.linspace(-1.1, 1.1, 80), np.linspace(-1.1, 1.1, 80))
    grid = np.stack([gx.ravel(), gy.ravel()], axis=1).astype(np.float32)
    gp = model(ox.from_numpy(grid)).softmax(-1).numpy()[:, 1].reshape(gx.shape)
    ax1.contourf(gx, gy, gp, levels=20, cmap="coolwarm", alpha=0.75)
    ax1.scatter(X_np[:, 0], X_np[:, 1], c=y_cls, cmap="coolwarm", edgecolors="k", s=24)
    ax1.set_title(f"Oxideon Spiral Boundary ({accs[-1]*100:.1f}% acc)")
    ax2.plot(losses, label="Loss", color="#10b981")
    ax2.plot(accs, label="Accuracy", color="#3b82f6")
    ax2.set_title("Training Curve (AdamW)")
    ax2.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib to render decision boundary & loss plots.")

## 4. CNN (`Conv2d`), Transformer (`MultiheadAttention`), Quantization & SafeTensors

In [ ]:
# Conv2d + MultiheadAttention
conv = ox.nn.Conv2d(3, 16, kernel_size=3, padding=1)
feat = conv(ox.randn([2, 3, 16, 16])).gelu()
print("Conv2d output shape:", list(feat.shape))

mha = ox.nn.MultiheadAttention(embed_dim=32, num_heads=4)
tokens = ox.randn([2, 8, 32])
attn_out, attn_weights = mha(tokens, tokens, tokens)
print("MultiheadAttention output:", list(attn_out.shape), "| weights:", list(attn_weights.shape))

# Dynamic Quantization + SafeTensors save_file / load_file
seq = ox.nn.Sequential(ox.nn.Linear(16, 64), ox.nn.Linear(64, 4))
qseq = ox.quantize_dynamic(seq)
q_out = qseq(ox.randn([4, 16]))
print("Quantized Sequential output:", list(q_out.shape))

with tempfile.TemporaryDirectory() as tmpdir:
    ckpt_path = os.path.join(tmpdir, "spiral_weights.safetensors")
    ox.save_file({f"p{i}": p.detach() for i, p in enumerate(model.parameters())}, ckpt_path)
    reloaded = ox.load_file(ckpt_path)
    print(f"Saved & reloaded {len(reloaded)} parameter tensors cleanly via SafeTensors.")

## 5. Scientific Computing: Linear Algebra (`qr`, `svd`), `fft.rfft` & Distributions

In [ ]:
M = ox.Tensor.from_slice([3.0, 1.0, 1.0, 3.0], [2, 2])
Q, R = ox.linalg.qr(M)
U, S, Vt = ox.linalg.svd(M)
print("QR shapes:", list(Q.shape), list(R.shape), "| singular values:", [round(v, 4) for v in S.to_list()])

sig = ox.Tensor.from_slice([math.sin(2 * math.pi * k / 16.0) for k in range(16)], [16])
spec = ox.fft.rfft(sig)
print("rfft spectrum shape:", list(spec.shape))

dist = ox.distributions.Normal(0.0, 1.0)
samples = dist.sample([1000]).numpy()
print(f"Normal(0,1) 1k sample mean={samples.mean():.3f}, std={samples.std():.3f}")

## 6. Zero-Copy NumPy / PyTorch DLPack Interop & CPU Speed Benchmark

In [ ]:
x_np = ox.randn([32, 128, 768]).numpy()
xo = ox.from_numpy(x_np)
for _ in range(3):
    _ = xo.softmax(-1)
t0 = time.perf_counter()
for _ in range(25):
    _ = xo.softmax(-1)
ox_ms = (time.perf_counter() - t0) / 25 * 1000

try:
    import torch
    xt = torch.randn(32, 128, 768)
    xo_dl = ox.from_dlpack(xt)
    xt_rt = torch.utils.dlpack.from_dlpack(xo_dl)
    assert xt_rt.data_ptr() == xt.data_ptr()
    for _ in range(3):
        _ = torch.softmax(xt, -1)
    t0 = time.perf_counter()
    for _ in range(25):
        _ = torch.softmax(xt, -1)
    pt_ms = (time.perf_counter() - t0) / 25 * 1000
    print(f"Zero-copy DLPack verified! softmax [32,128,768]: Oxideon={ox_ms:.2f} ms, PyTorch={pt_ms:.2f} ms")
except ImportError:
    print(f"softmax [32,128,768]: Oxideon={ox_ms:.2f} ms (install torch to compare)")